# ChronosCast — Notebook 02: Time Series Trend & Seasonality Diagnostics
**Project**: ChronosCast: Intelligent Time Series Trend Analysis & Forecasting Platform

### Notebook Scope:
1. Long-term trend analysis with rolling moving averages and rolling volatility
2. Multi-granularity seasonality: Day of week, month, quarter, annual growth
3. Classical time series decomposition (Observed, Trend, Seasonal, Residual)
4. Stationarity verification using Augmented Dickey-Fuller (ADF) & KPSS tests
5. Autocorrelation Function (ACF) & Partial Autocorrelation Function (PACF)


In [ ]:
import pandas as pd
import numpy as np
from chronoscast.analysis.eda import TimeSeriesEDA
from chronoscast.analysis.diagnostics import run_stationarity_tests, compute_acf_pacf, perform_decomposition

df = pd.read_parquet("../data/interim/cleaned_sales.parquet")
eda = TimeSeriesEDA(df)
daily = eda.daily_aggregated_sales()
daily.head()


### 1. Stationarity Tests (ADF & KPSS)

In [ ]:
# Test on aggregate daily demand
agg_series = daily.set_index("date")["sales"]
diag_agg = run_stationarity_tests(agg_series)
print("Aggregate Daily Demand Stationarity:")
print(f"  ADF Statistic: {diag_agg['adf']['statistic']} (p-value: {diag_agg['adf']['p_value']}) -> Stationary: {diag_agg['adf']['is_stationary']}")
print(f"  KPSS Statistic: {diag_agg['kpss']['statistic']} (p-value: {diag_agg['kpss']['p_value']}) -> Stationary: {diag_agg['kpss']['is_stationary']}")
print(f"  Conclusion: {diag_agg['conclusion']}")


### 2. Time Series Decomposition

In [ ]:
decomp = perform_decomposition(agg_series, model="additive", period=7)
print("Decomposition successful.")
print(f"Trend mean: {decomp['trend'].mean():.2f}")
print(f"Seasonal amplitude: {decomp['seasonal'].max() - decomp['seasonal'].min():.2f}")
print(f"Residual std: {decomp['resid'].std():.2f}")


### 3. Autocorrelation & Partial Autocorrelation Diagnostics

In [ ]:
s1i1 = df[(df["store"] == 1) & (df["item"] == 1)].sort_values("date").set_index("date")["sales"]
acf_res = compute_acf_pacf(s1i1, nlags=14)
print("Lags 1 to 14 ACF values for Store 1, Item 1:")
for lag, val in zip(acf_res["lags"][1:15], acf_res["acf"][1:15]):
    print(f"  Lag {lag:2d}: {val:.4f}")
